In [ ]:
# === Setup ===
# Runtime: ~5 phút trên Colab T4
# Hardware: Cần GPU (hoặc CPU nếu chạy model nhỏ)
import random, numpy as np
import torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)
# Cẩm nang P08: Cho phép torch, transformers, torchvision


# Từ điển: Code LoRA (Low-Rank Adaptation) từ con số 0

Thay vì huấn luyện toàn bộ trọng số `W` của một layer (rất lớn), ta huấn luyện hai ma trận nhỏ `A` và `B` sao cho $\Delta W = B \times A$.


In [ ]:
import torch
import torch.nn as nn

# Một Linear Layer thông thường
class SimpleLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.randn(out_features))
        
    def forward(self, x):
        return x @ self.weight.T + self.bias

# Lớp bọc LoRA
class LoRALinear(nn.Module):
    def __init__(self, linear_layer, rank=4, alpha=8):
        super().__init__()
        self.in_features = linear_layer.weight.shape[1]
        self.out_features = linear_layer.weight.shape[0]
        
        # Đóng băng layer gốc
        self.base_layer = linear_layer
        for param in self.base_layer.parameters():
            param.requires_grad = False
            
        # Khởi tạo ma trận A và B
        # A: thu nhỏ chiều (in_features -> rank)
        self.lora_A = nn.Parameter(torch.randn(rank, self.in_features) / self.in_features)
        # B: phóng to chiều (rank -> out_features), khởi tạo bằng 0
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank))
        self.scaling = alpha / rank
        
    def forward(self, x):
        base_out = self.base_layer(x)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out

# Demo
linear = SimpleLinear(1024, 1024)
lora_linear = LoRALinear(linear, rank=8)

x = torch.randn(1, 1024)
print("Base shape:", linear(x).shape)
